# B Cell Population Analysis - Quick Overview

**Author**: r2end  
**Date**: 2026-01-14  
**Purpose**: Comprehensive analysis of B cell populations from scRNA-seq data

## Analysis Pipeline
1. Data loading and structure validation
2. Quality metrics overview
3. UMAP visualization (batch, cell type, key markers)
4. Cell type composition analysis
5. Marker gene expression patterns
6. (Optional) Differential expression analysis

**Input**: `adata_bcell_FINAL.h5ad`  
**Expected structure**:
- `.X`: log1p normalized
- `.layers['counts']`: raw counts
- `.layers['log1p']`: log1p normalized
- `.raw`: full gene matrix (if available)
- `.obs`: metadata with cell type annotations
- `.obsm`: UMAP, PCA coordinates

## 0. Setup and Configuration

In [ ]:
# Import libraries
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(8, 6))
sc.settings.n_jobs = 48  # Multi-core processing

print("Libraries loaded successfully")

In [ ]:
# Configuration parameters - ADJUST THESE BASED ON YOUR DATA
INPUT_FILE = '/home/h2048/data/py/0111/celltypist_bcell/adata_bcell_FINAL.h5ad'
OUTPUT_DIR = '/home/h2048/data/py/0114/celltypist_bcell/analysis_outputs'
FIG_DIR = f'{OUTPUT_DIR}/figures'

# Create output directories
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Key annotation columns - ADJUST THESE BASED ON YOUR DATA
BATCH_KEY = 'sample'  # or 'batch', 'donor', 'patient_id'
CELLTYPE_KEY = 'cell_type_scanvi_filt'  # or 'predicted_labels', 'leiden', 'cluster'
DISEASE_KEY = 'disease_status'  # or 'group', 'condition'

print(f"Configuration complete")
print(f"  Input: {INPUT_FILE}")
print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# Define B cell marker genes
BCELL_MARKERS = {
    'Pan_B': ['CD19', 'MS4A1', 'CD79A', 'CD79B'],  # MS4A1 = CD20
    'Naive_B': ['TCL1A', 'FCER2', 'IL4R'],
    'Memory_B': ['CD27', 'TNFRSF13B'],  # TNFRSF13B = TACI
    'Plasma': ['MZB1', 'JCHAIN', 'IGHA1', 'IGHG1', 'XBP1', 'SDC1'],  # SDC1 = CD138
    'Activated_B': ['CD69', 'CD83', 'BCL2A1'],
    'GC_B': ['AICDA', 'BCL6', 'CXCR4']  # Germinal center B cells
}

# Flatten marker list
ALL_MARKERS = [gene for genes in BCELL_MARKERS.values() for gene in genes]

print("B cell markers defined:")
for category, markers in BCELL_MARKERS.items():
    print(f"  {category}: {', '.join(markers)}")

## 1. Data Loading and Structure Validation

In [ ]:
# Load data
print("Loading data...")
adata = sc.read_h5ad(INPUT_FILE)

print(f"\n{'='*80}")
print("DATA STRUCTURE OVERVIEW")
print(f"{'='*80}")
print(f"Shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"\nLayers available: {list(adata.layers.keys())}")
print(f"Has .raw: {adata.raw is not None}")
if adata.raw is not None:
    print(f"  .raw shape: {adata.raw.shape[0]:,} cells × {adata.raw.shape[1]:,} genes")

print(f"\n.obsm keys (embeddings): {list(adata.obsm.keys())}")
print(f".obsp keys (graphs): {list(adata.obsp.keys())}")

In [ ]:
# Check metadata columns
print(f"\n{'='*80}")
print("METADATA COLUMNS")
print(f"{'='*80}")
print(f"Total columns: {len(adata.obs.columns)}")
print("\nKey columns:")
for col in adata.obs.columns[:20]:  # Show first 20
    unique_vals = adata.obs[col].nunique()
    print(f"  - {col}: {unique_vals} unique values")

In [ ]:
# Validate expected columns
print(f"\n{'='*80}")
print("COLUMN VALIDATION")
print(f"{'='*80}")

expected_cols = {
    'Batch/Sample': BATCH_KEY,
    'Cell Type': CELLTYPE_KEY,
    'Disease Status': DISEASE_KEY
}

for name, col in expected_cols.items():
    if col in adata.obs.columns:
        n_unique = adata.obs[col].nunique()
        print(f"✓ {name} ({col}): {n_unique} categories")
        if n_unique <= 20:
            print(f"  Values: {adata.obs[col].unique().tolist()}")
    else:
        print(f"⚠ {name} ({col}): NOT FOUND")
        print(f"  Available columns with similar names:")
        similar = [c for c in adata.obs.columns if any(keyword in c.lower() 
                  for keyword in col.lower().split('_'))]
        print(f"    {similar}")

## 2. Quality Metrics Overview

In [ ]:
# Basic QC metrics
print(f"\n{'='*80}")
print("QUALITY CONTROL METRICS")
print(f"{'='*80}")

qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt']
available_qc = [col for col in qc_metrics if col in adata.obs.columns]

if available_qc:
    for metric in available_qc:
        values = adata.obs[metric]
        print(f"\n{metric}:")
        print(f"  Mean: {values.mean():.2f}")
        print(f"  Median: {values.median():.2f}")
        print(f"  Range: [{values.min():.2f}, {values.max():.2f}]")
else:
    print("⚠ Standard QC metrics not found. Data might be already filtered.")
    print(f"  Available obs columns: {adata.obs.columns.tolist()}")

In [ ]:
# QC visualization (if metrics available)
if available_qc:
    fig, axes = plt.subplots(1, len(available_qc), figsize=(5*len(available_qc), 4))
    if len(available_qc) == 1:
        axes = [axes]
    
    for ax, metric in zip(axes, available_qc):
        sc.pl.violin(adata, metric, ax=ax, show=False)
        ax.set_title(f'{metric} Distribution', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/01_qc_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/01_qc_metrics.png")
else:
    print("⚠ Skipping QC visualization (metrics not available)")

## 3. UMAP Visualization - Batch and Cell Types

In [ ]:
# Check if UMAP exists
if 'X_umap' not in adata.obsm.keys():
    print("⚠ UMAP not found. Computing UMAP...")
    if 'X_pca' not in adata.obsm.keys():
        print("  Computing PCA first...")
        sc.pp.pca(adata, n_comps=50)
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
    sc.tl.umap(adata)
    print("✓ UMAP computed")
else:
    print("✓ UMAP coordinates found")

In [ ]:
# UMAP by batch
if BATCH_KEY in adata.obs.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(adata, color=BATCH_KEY, ax=ax, show=False, 
               title='B Cells - Batch Distribution', legend_loc='on data')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/02_umap_batch.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/02_umap_batch.png")
else:
    print(f"⚠ Batch column '{BATCH_KEY}' not found. Skipping batch UMAP.")

In [ ]:
# UMAP by cell type
if CELLTYPE_KEY in adata.obs.columns:
    fig, ax = plt.subplots(figsize=(12, 8))
    sc.pl.umap(adata, color=CELLTYPE_KEY, ax=ax, show=False,
               title='B Cells - Cell Type Annotation', legend_fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/03_umap_celltype.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/03_umap_celltype.png")
    
    # Print cell type distribution
    print(f"\n{'='*60}")
    print("CELL TYPE DISTRIBUTION")
    print(f"{'='*60}")
    celltype_counts = adata.obs[CELLTYPE_KEY].value_counts()
    for ct, count in celltype_counts.items():
        pct = 100 * count / len(adata)
        print(f"{ct:30s}: {count:6,} cells ({pct:5.2f}%)")
else:
    print(f"⚠ Cell type column '{CELLTYPE_KEY}' not found. Skipping cell type UMAP.")

In [ ]:
# UMAP by disease status (if available)
if DISEASE_KEY in adata.obs.columns:
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(adata, color=DISEASE_KEY, ax=ax, show=False,
               title='B Cells - Disease Status', palette='Set2')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/04_umap_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/04_umap_disease.png")
else:
    print(f"⚠ Disease column '{DISEASE_KEY}' not found. Skipping disease UMAP.")

## 4. B Cell Marker Gene Expression

In [ ]:
# Check marker availability
use_raw = adata.raw is not None
gene_names = adata.raw.var_names if use_raw else adata.var_names

available_markers = {}
for category, markers in BCELL_MARKERS.items():
    available = [m for m in markers if m in gene_names]
    if available:
        available_markers[category] = available

print(f"\n{'='*80}")
print("MARKER GENE AVAILABILITY")
print(f"{'='*80}")
print(f"Searching in: {'.raw' if use_raw else '.X'} ({len(gene_names):,} genes)")
for category, markers in BCELL_MARKERS.items():
    available = available_markers.get(category, [])
    missing = set(markers) - set(available)
    print(f"\n{category}:")
    print(f"  Available ({len(available)}/{len(markers)}): {', '.join(available)}")
    if missing:
        print(f"  Missing: {', '.join(missing)}")

In [ ]:
# UMAP colored by key markers
plot_markers = []
for markers in available_markers.values():
    plot_markers.extend(markers[:2])  # Take first 2 from each category

if plot_markers:
    n_markers = len(plot_markers)
    n_cols = 4
    n_rows = (n_markers + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]
    
    for i, marker in enumerate(plot_markers):
        sc.pl.umap(adata, color=marker, ax=axes[i], show=False, 
                   use_raw=use_raw, cmap='viridis', title=marker, 
                   frameon=False, vmin=0)
    
    # Hide unused axes
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/05_umap_markers.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/05_umap_markers.png")
else:
    print("⚠ No B cell markers found in dataset")

In [ ]:
# Dotplot for all available markers (by cell type)
if available_markers and CELLTYPE_KEY in adata.obs.columns:
    all_available = [m for markers in available_markers.values() for m in markers]
    
    # Check if we have enough cells per cell type
    celltype_counts = adata.obs[CELLTYPE_KEY].value_counts()
    valid_celltypes = celltype_counts[celltype_counts >= 10].index.tolist()
    
    if len(valid_celltypes) > 0:
        adata_subset = adata[adata.obs[CELLTYPE_KEY].isin(valid_celltypes)].copy()
        
        fig, ax = plt.subplots(figsize=(12, max(6, len(valid_celltypes)*0.4)))
        sc.pl.dotplot(adata_subset, all_available, groupby=CELLTYPE_KEY, 
                      ax=ax, show=False, use_raw=use_raw, standard_scale='var',
                      title='B Cell Marker Expression Across Cell Types')
        plt.tight_layout()
        plt.savefig(f'{FIG_DIR}/06_dotplot_markers.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"✓ Saved: {FIG_DIR}/06_dotplot_markers.png")
    else:
        print("⚠ Not enough cells per cell type for dotplot (need ≥10 cells)")
else:
    print("⚠ Skipping dotplot (no markers or cell types)")

## 5. Cell Type Composition Analysis

In [ ]:
# Cell type proportion by batch/sample
if CELLTYPE_KEY in adata.obs.columns and BATCH_KEY in adata.obs.columns:
    # Calculate proportions
    prop_df = adata.obs.groupby([BATCH_KEY, CELLTYPE_KEY]).size().unstack(fill_value=0)
    prop_df = prop_df.div(prop_df.sum(axis=1), axis=0) * 100
    
    # Heatmap
    fig, ax = plt.subplots(figsize=(12, max(6, len(prop_df)*0.3)))
    sns.heatmap(prop_df, annot=True, fmt='.1f', cmap='YlOrRd', 
                cbar_kws={'label': 'Percentage (%)'}, ax=ax)
    ax.set_title('B Cell Composition Across Samples (%)', fontsize=14)
    ax.set_xlabel('Cell Type', fontsize=12)
    ax.set_ylabel(BATCH_KEY, fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/07_composition_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/07_composition_heatmap.png")

In [ ]:
# Stacked bar plot
if CELLTYPE_KEY in adata.obs.columns and BATCH_KEY in adata.obs.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    prop_df.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
    ax.set_title('B Cell Composition Across Samples', fontsize=14)
    ax.set_xlabel(BATCH_KEY, fontsize=12)
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/08_composition_barplot.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/08_composition_barplot.png")

In [ ]:
# Cell type proportion by disease status (if available)
if CELLTYPE_KEY in adata.obs.columns and DISEASE_KEY in adata.obs.columns:
    # Calculate proportions
    prop_disease = adata.obs.groupby([DISEASE_KEY, CELLTYPE_KEY]).size().unstack(fill_value=0)
    prop_disease = prop_disease.div(prop_disease.sum(axis=1), axis=0) * 100
    
    # Bar plot comparison
    fig, ax = plt.subplots(figsize=(12, 6))
    prop_disease.T.plot(kind='bar', ax=ax, width=0.8)
    ax.set_title('B Cell Composition by Disease Status', fontsize=14)
    ax.set_xlabel('Cell Type', fontsize=12)
    ax.set_ylabel('Percentage (%)', fontsize=12)
    ax.legend(title='Disease Status', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/09_composition_disease.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/09_composition_disease.png")
    
    # Print summary
    print(f"\n{'='*80}")
    print("CELL TYPE PROPORTION BY DISEASE STATUS")
    print(f"{'='*80}")
    print(prop_disease.to_string())

## 6. Summary Statistics

In [ ]:
print(f"\n{'='*80}")
print("ANALYSIS SUMMARY")
print(f"{'='*80}")
print(f"Total cells analyzed: {adata.shape[0]:,}")
print(f"Total genes: {adata.shape[1]:,}")
if adata.raw is not None:
    print(f"Full gene matrix: {adata.raw.shape[1]:,} genes")

if BATCH_KEY in adata.obs.columns:
    print(f"\nNumber of batches/samples: {adata.obs[BATCH_KEY].nunique()}")
    
if CELLTYPE_KEY in adata.obs.columns:
    print(f"\nNumber of cell types: {adata.obs[CELLTYPE_KEY].nunique()}")
    print("\nTop 5 cell types:")
    for i, (ct, count) in enumerate(adata.obs[CELLTYPE_KEY].value_counts().head(5).items(), 1):
        pct = 100 * count / len(adata)
        print(f"  {i}. {ct}: {count:,} cells ({pct:.2f}%)")

if DISEASE_KEY in adata.obs.columns:
    print(f"\nDisease status distribution:")
    for status, count in adata.obs[DISEASE_KEY].value_counts().items():
        pct = 100 * count / len(adata)
        print(f"  - {status}: {count:,} cells ({pct:.2f}%)")

print(f"\n{'='*80}")
print(f"All figures saved to: {FIG_DIR}")
print(f"{'='*80}")

## 7. Optional: Marker Gene Analysis

This section computes marker genes for each cell type. It may take several minutes for large datasets.

In [ ]:
# Find top marker genes for each cell type
if CELLTYPE_KEY in adata.obs.columns:
    print("\n[OPTIONAL] Computing marker genes for each cell type...")
    print("This may take a few minutes for large datasets...")
    
    # Check if markers already computed
    if 'rank_genes_groups' not in adata.uns.keys():
        try:
            sc.tl.rank_genes_groups(
                adata, 
                groupby=CELLTYPE_KEY, 
                method='wilcoxon',
                use_raw=use_raw,
                n_genes=100
            )
            print("✓ Marker gene computation complete")
            
            # Save top markers to CSV
            marker_df = sc.get.rank_genes_groups_df(adata, group=None)
            marker_df.to_csv(f'{OUTPUT_DIR}/marker_genes_all_celltypes.csv', index=False)
            print(f"✓ Saved: {OUTPUT_DIR}/marker_genes_all_celltypes.csv")
            
        except Exception as e:
            print(f"⚠ Marker gene computation failed: {e}")
    else:
        print("✓ Marker genes already computed (found in adata.uns)")

In [ ]:
# Visualize top markers
if 'rank_genes_groups' in adata.uns.keys():
    fig = sc.pl.rank_genes_groups_dotplot(
        adata, 
        n_genes=5, 
        use_raw=use_raw,
        show=False,
        return_fig=True
    )
    fig.savefig(f'{FIG_DIR}/10_top_markers_dotplot.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved: {FIG_DIR}/10_top_markers_dotplot.png")

In [ ]:
# Compute marker genes for each cell type
print("Computing marker genes...")

# Use raw counts for differential expression
use_raw = adata.raw is not None

# Compute markers using Wilcoxon test
sc.tl.rank_genes_groups(
    adata, 
    groupby=CELLTYPE_KEY, 
    method='wilcoxon',
    use_raw=use_raw,
    n_genes=50,
    key_added='rank_genes_groups'
)

# Display top 10 markers for each cell type
print(f"\nTop 10 marker genes per cell type:")
print("="*80)
for celltype in adata.obs[CELLTYPE_KEY].unique():
    markers = sc.get.rank_genes_groups_df(adata, group=celltype, key='rank_genes_groups')
    top_genes = markers.head(10)['names'].tolist()
    print(f"\n{celltype}:")
    print(f"  {', '.join(top_genes)}")

# Save all markers to CSV
marker_df = sc.get.rank_genes_groups_df(adata, group=None, key='rank_genes_groups')
output_file = f'{OUTPUT_DIR}/marker_genes_by_celltype.csv'
marker_df.to_csv(output_file, index=False)
print(f"\n✓ All markers saved to: {output_file}")

# Quick dotplot visualization
fig, ax = plt.subplots(figsize=(10, max(6, adata.obs[CELLTYPE_KEY].nunique()*0.4)))
sc.pl.rank_genes_groups_dotplot(adata, n_genes=5, key='rank_genes_groups', 
                                 use_raw=use_raw, show=False, ax=ax)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/marker_genes_dotplot.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Dotplot saved to: {FIG_DIR}/marker_genes_dotplot.png")

In [ ]:

B_COL = "cell_type_scanvi_filt"          # 改成你的列名
B_LABEL = "B cells"           # 只看这一类

adata_b = adata[adata.obs[B_COL] == B_LABEL].copy()

gene_sets = {
    "naive":  ["TCL1A","IGHM","IGHD","FCER2","CR2","CD72"],
    "memory": ["CD27","TNFRSF13B","AIM2","BANK1","GPR183","CD44","MS4A1"],
    "gc":     ["AICDA","BCL6","RGS13","SERPINA9","MEF2B","CD38","IRF8"],
    "plasma": ["XBP1","MZB1","PRDM1","JCHAIN","SDC1","TNFRSF17","TENT5C"],
    "atypical_memory": ["FCRL4","FCRL5","ITGAX","TBX21","ZEB2","CCR6"],
}

# 确保基因存在
varnames = set(adata_b.var_names)
for k, gs in gene_sets.items():
    gs2 = [g for g in gs if g in varnames]
    if len(gs2) >= 3:
        sc.tl.score_genes(adata_b, gs2, score_name=f"score_{k}", use_raw=False)
    else:
        adata_b.obs[f"score_{k}"] = np.nan

score_cols = [c for c in adata_b.obs.columns if c.startswith("score_")]
adata_b.obs["best_program"] = adata_b.obs[score_cols].idxmax(axis=1).str.replace("score_", "", regex=False)

# 看各program占比
print(adata_b.obs["best_program"].value_counts(normalize=True).round(3))

# 画关键marker（强烈建议）
sc.pl.dotplot(
    adata_b,
    var_names={
        "Naive": ["TCL1A","FCER2","IGHM","IGHD","CR2"],
        "Memory": ["CD27","TNFRSF13B","AIM2","BANK1","GPR183","IGHG1","IGHA1"],
        "GC": ["AICDA","BCL6","RGS13","SERPINA9","CD38"],
        "Plasma": ["XBP1","MZB1","PRDM1","JCHAIN","SDC1"],
        "Atypical": ["FCRL4","FCRL5","ITGAX","TBX21","CCR6"]
    },
    groupby="best_program",
    standard_scale="var",
)

## 8. Analysis Complete

All outputs have been saved to the specified directories.

In [ ]:
print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Figures: {FIG_DIR}")
print("\nGenerated outputs:")
print("  - QC metrics visualization")
print("  - UMAP plots (batch, cell type, disease, markers)")
print("  - Marker gene dotplot")
print("  - Cell composition heatmap and barplots")
print("  - (Optional) Top marker genes per cell type")
print("\nNext steps:")
print("  1. Review UMAP plots for batch effects and cell type separation")
print("  2. Validate marker gene expression patterns")
print("  3. Perform differential expression if comparing conditions")
print("  4. Consider trajectory analysis for B cell maturation")
print("="*80)

In [ ]:
# ============================================================================
# CELL 1: Load data and check current annotations
# ============================================================================
import scanpy as sc
import numpy as np
import pandas as pd
import scvi
import torch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration
INPUT_FILE = "/home/h2048/data/py/0111/celltypist_bcell/adata_bcell_FINAL.h5ad"
OUTPUT_DIR = Path("/home/h2048/data/py/0111/celltypist_bcell")
MODEL_DIR = OUTPUT_DIR / "models"
FIG_DIR = OUTPUT_DIR / "figures"

# Parameters (from original pipeline)
BATCH_KEY = 'sample'
N_LATENT = 50
N_LAYERS = 2
SCANVI_MAX_EPOCHS = 200
BATCH_SIZE = 2048
LEARNING_RATE = 1e-3
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 50

# GPU
GPU_AVAILABLE = torch.cuda.is_available()
accelerator = 'gpu' if GPU_AVAILABLE else 'cpu'
devices = 1 if GPU_AVAILABLE else 'auto'

print(f"Loading data: {INPUT_FILE}")
# adata = sc.read_h5ad(INPUT_FILE)

print(f"\nData loaded:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

print(f"\nCurrent cell_type_scanvi_filt distribution:")
print(adata.obs['cell_type_scanvi_filt'].value_counts())

In [ ]:
# ============================================================================
# CELL 2: Modify annotations - B cells -> Memory B cells
# ============================================================================

print("="*80)
print("MODIFYING ANNOTATIONS: B cells -> Memory B cells")
print("="*80)

# Backup original
adata.obs['cell_type_scanvi_filt_original'] = adata.obs['cell_type_scanvi_filt'].copy()

# Modify B cells -> Memory B cells
mask = adata.obs['cell_type_scanvi_filt'] == 'B cells'
n_modified = mask.sum()

adata.obs.loc[mask, 'cell_type_scanvi_filt'] = 'Memory B cells'

print(f"\n✓ Modified {n_modified:,} cells: 'B cells' -> 'Memory B cells'")

# Update labels_for_scanvi (this is what scANVI will use for training)
adata.obs['labels_for_scanvi'] = adata.obs['cell_type_scanvi_filt'].astype('category')

# Ensure 'Unknown' category exists
if 'Unknown' not in adata.obs['labels_for_scanvi'].cat.categories:
    adata.obs['labels_for_scanvi'] = adata.obs['labels_for_scanvi'].cat.add_categories(['Unknown'])

print(f"\n✓ Updated labels_for_scanvi")
print(f"\nNew cell_type_scanvi_filt distribution:")
print(adata.obs['cell_type_scanvi_filt'].value_counts())

# Statistics
n_unknown = (adata.obs['labels_for_scanvi'] == 'Unknown').sum()
n_labeled = adata.n_obs - n_unknown
n_unique = adata.obs['labels_for_scanvi'].nunique() - 1  # Exclude Unknown

print(f"\nscANVI training labels:")
print(f"  Labeled: {n_labeled:,} ({n_labeled/adata.n_obs*100:.1f}%)")
print(f"  Unknown: {n_unknown:,} ({n_unknown/adata.n_obs*100:.1f}%)")
print(f"  Unique types: {n_unique}")

In [ ]:
# ============================================================================
# CELL 3: Rebuild adata_model for scANVI retraining
# ============================================================================

print("="*80)
print("REBUILDING adata_model FOR scANVI")
print("="*80)

# Load HVG mask from saved gene list
hvg_file = MODEL_DIR / "hvg_genes.txt"
if not hvg_file.exists():
    raise FileNotFoundError(f"HVG gene list not found: {hvg_file}")

hvg_genes = pd.read_csv(hvg_file, header=None)[0].astype(str).tolist()
print(f"\n✓ Loaded HVG gene list: {len(hvg_genes)} genes")

# Create HVG mask
hvg_mask = adata.var_names.isin(hvg_genes)
# Convert to numpy if needed
if hasattr(hvg_mask, 'to_numpy'):
    hvg_mask = hvg_mask.to_numpy()
elif hasattr(hvg_mask, 'values'):
    hvg_mask = hvg_mask.values
# else it's already numpy array

n_hvg = hvg_mask.sum()

print(f"✓ HVG mask created: {n_hvg:,} genes")

# Build adata_model (HVG-only subset)
print(f"\nBuilding adata_model...")
X_hvg = adata.layers['counts'][:, hvg_mask].copy()
obs_hvg = adata.obs[[BATCH_KEY, 'labels_for_scanvi']].copy()
var_hvg = adata.var.loc[hvg_mask, []].copy()

adata_model = sc.AnnData(X=X_hvg, obs=obs_hvg, var=var_hvg)
adata_model.layers['counts'] = adata_model.X.copy()

print(f"\n✓ adata_model created:")
print(f"  Shape: {adata_model.n_obs:,} cells × {adata_model.n_vars:,} genes")
print(f"  obs columns: {list(adata_model.obs.columns)}")

# Verify labels
print(f"\n✓ Labels verification:")
print(f"  Column 'labels_for_scanvi' exists: {'labels_for_scanvi' in adata_model.obs.columns}")
print(f"  Unique labels: {adata_model.obs['labels_for_scanvi'].nunique()}")

In [ ]:
# ============================================================================
# CELL 4: Load pretrained scVI and retrain scANVI
# ============================================================================

print("="*80)
print("RETRAINING scANVI WITH CORRECTED LABELS")
print("="*80)

# Load scVI model
scvi_model_dir = MODEL_DIR / "scvi_model"
print(f"\nLoading pretrained scVI: {scvi_model_dir}")

scvi.model.SCVI.setup_anndata(
    adata_model,
    layer='counts',
    batch_key=BATCH_KEY
)

vae = scvi.model.SCVI.load(str(scvi_model_dir), adata=adata_model)
print(f"✓ scVI model loaded")

# Initialize scANVI from scVI
print(f"\nInitializing scANVI with corrected labels...")
lvae = scvi.model.SCANVI.from_scvi_model(
    vae,
    adata=adata_model,
    labels_key='labels_for_scanvi',
    unlabeled_category='Unknown'
)
print(f"✓ scANVI initialized")

# Train scANVI
print(f"\n{'='*80}")
print(f"Training scANVI...")
print(f"{'='*80}\n")

import time
start_time = time.time()

train_kwargs = {
    'max_epochs': SCANVI_MAX_EPOCHS,
    'batch_size': BATCH_SIZE,
    'train_size': 0.9,
    'accelerator': accelerator,
    'devices': devices,
    'plan_kwargs': {'lr': LEARNING_RATE},
}
if EARLY_STOPPING:
    train_kwargs['early_stopping'] = True
    train_kwargs['early_stopping_patience'] = EARLY_STOPPING_PATIENCE

lvae.train(**train_kwargs)

elapsed = time.time() - start_time
print(f"\n✓ Training complete: {int(elapsed//60)}m {int(elapsed%60)}s")

# Save updated model
scanvi_model_dir_new = MODEL_DIR / "scanvi_model_corrected"
print(f"\nSaving corrected scANVI model: {scanvi_model_dir_new}")
lvae.save(scanvi_model_dir_new, overwrite=True)
print(f"✓ Model saved")

In [ ]:
# ============================================================================
# CELL 5: Generate new predictions and update adata
# ============================================================================

print("="*80)
print("UPDATING PREDICTIONS")
print("="*80)

# Generate predictions (index-aligned)
print(f"\nGenerating scANVI predictions...")

pred = pd.Series(lvae.predict(), index=adata_model.obs_names)
adata.obs['cell_type_scanvi_corrected'] = pred.reindex(adata.obs_names).astype(str).values

# Confidence scores
probs = np.asarray(lvae.predict(soft=True))
conf = pd.Series(probs.max(axis=1), index=adata_model.obs_names)
adata.obs['scanvi_confidence_corrected'] = conf.reindex(adata.obs_names).values

# Latent representation
z = pd.DataFrame(lvae.get_latent_representation(), index=adata_model.obs_names)
adata.obsm['X_scanvi_corrected'] = z.reindex(adata.obs_names).to_numpy()

print(f"✓ Predictions updated:")
print(f"  cell_type_scanvi_corrected: {adata.obs['cell_type_scanvi_corrected'].nunique()} unique types")
print(f"  Mean confidence: {adata.obs['scanvi_confidence_corrected'].mean():.3f}")
print(f"  X_scanvi_corrected: {adata.obsm['X_scanvi_corrected'].shape}")

# Compare old vs new
print(f"\n{'='*60}")
print("COMPARISON: Original vs Corrected")
print("="*60)

print(f"\nOriginal scANVI predictions:")
print(adata.obs['cell_type_scanvi_filt_original'].value_counts())

print(f"\nCorrected scANVI predictions:")
print(adata.obs['cell_type_scanvi_corrected'].value_counts())

# Cleanup
del adata_model
import gc
gc.collect()

In [ ]:
# ============================================================================
# CELL 6: Recompute UMAP on corrected scANVI latent space
# ============================================================================

print("="*80)
print("RECOMPUTING UMAP WITH CORRECTED scANVI")
print("="*80)

# Compute neighbors on corrected scANVI latent
print(f"\nComputing neighbors on X_scanvi_corrected...")
sc.pp.neighbors(
    adata,
    use_rep='X_scanvi_corrected',
    n_neighbors=15,
    key_added='neighbors_scanvi_corrected'
)

# Compute UMAP
print(f"Computing UMAP...")
sc.tl.umap(adata, neighbors_key='neighbors_scanvi_corrected')
adata.obsm['X_umap_scanvi_corrected'] = adata.obsm['X_umap'].copy()

print(f"✓ UMAP computed: {adata.obsm['X_umap_scanvi_corrected'].shape}")

# Generate comparison figure
print(f"\nGenerating comparison figure...")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Original annotations
sc.pl.embedding(
    adata,
    basis='umap_scanvi_corrected',
    color='cell_type_scanvi_filt_original',
    ax=axes[0],
    show=False,
    title='Original scANVI (with "B cells")',
    size=2,
    legend_loc='right margin',
    frameon=False
)

# Corrected annotations (input labels)
sc.pl.embedding(
    adata,
    basis='umap_scanvi_corrected',
    color='cell_type_scanvi_filt',
    ax=axes[1],
    show=False,
    title='Corrected Labels (B cells → Memory B cells)',
    size=2,
    legend_loc='right margin',
    frameon=False
)

# New scANVI predictions
sc.pl.embedding(
    adata,
    basis='umap_scanvi_corrected',
    color='cell_type_scanvi_corrected',
    ax=axes[2],
    show=False,
    title='Retrained scANVI Predictions',
    size=2,
    legend_loc='right margin',
    frameon=False
)

plt.tight_layout()
output_fig = FIG_DIR / 'scanvi_correction_comparison.pdf'
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved: {output_fig}")

In [ ]:
# ============================================================================
# CELL 7: Save corrected results
# ============================================================================

print("="*80)
print("SAVING CORRECTED RESULTS")
print("="*80)

# Final output file
output_file = OUTPUT_DIR / "adata_bcell_FINAL_corrected_20260114.h5ad"

# Update metadata
adata.uns['correction_info'] = {
    'correction_date': '2026-01-14',
    'correction_type': 'B cells -> Memory B cells',
    'n_cells_modified': int(mask.sum()),
    'scanvi_retrained': True,
    'new_model_path': str(scanvi_model_dir_new)
}

# Save
print(f"\nSaving to: {output_file}")
adata.write_h5ad(output_file, compression='gzip')

size_gb = output_file.stat().st_size / 1e9
print(f"✓ File saved: {size_gb:.2f} GB")

# Export corrected annotations
annotations_corrected = adata.obs[[
    'cell_type_scanvi_filt_original',  # Original
    'cell_type_scanvi_filt',            # Corrected input labels
    'cell_type_scanvi_corrected',       # New predictions
    'scanvi_confidence_corrected',
    BATCH_KEY
]].copy()

annotations_file = OUTPUT_DIR / "annotations_corrected.csv"
annotations_corrected.to_csv(annotations_file)
print(f"✓ Annotations exported: {annotations_file}")

# Save count statistics
from pathlib import Path
def save_counts(series, filename):
    df = pd.DataFrame({
        'cell_type': series.index,
        'count': series.values,
        'percentage': (series.values / series.sum() * 100).round(2)
    }).sort_values('count', ascending=False)
    df.to_csv(OUTPUT_DIR / filename, index=False)
    print(f"  ✓ {filename}")

print(f"\nSaving count statistics...")
save_counts(adata.obs['cell_type_scanvi_filt_original'].value_counts(), 
            'scanvi_counts_original.csv')
save_counts(adata.obs['cell_type_scanvi_corrected'].value_counts(), 
            'scanvi_counts_corrected.csv')

In [ ]:
# ============================================================================
# CELL 8: Final summary and validation
# ============================================================================

print("="*80)
print("🎉 CORRECTION COMPLETE - FINAL SUMMARY")
print("="*80)

print(f"\n📊 Changes:")
print(f"  Modified cells: {int(mask.sum()):,} ('B cells' → 'Memory B cells')")
print(f"  scANVI retrained: Yes")
print(f"  Training epochs: {SCANVI_MAX_EPOCHS}")

print(f"\n📈 Cell Type Distribution (Corrected):")
for ct, count in adata.obs['cell_type_scanvi_corrected'].value_counts().items():
    pct = 100 * count / adata.n_obs
    print(f"  {ct:30s}: {count:6,} cells ({pct:5.2f}%)")

print(f"\n🔬 Quality Metrics:")
print(f"  Mean confidence: {adata.obs['scanvi_confidence_corrected'].mean():.3f}")
print(f"  Cells analyzed: {adata.n_obs:,}")
print(f"  Unique types: {adata.obs['cell_type_scanvi_corrected'].nunique()}")

print(f"\n📁 Output Files:")
print(f"  h5ad (corrected): {output_file.name}")
print(f"  Model: {scanvi_model_dir_new}")
print(f"  Annotations: annotations_corrected.csv")
print(f"  Counts: scanvi_counts_corrected.csv")
print(f"  Figure: scanvi_correction_comparison.pdf")

print(f"\n✅ Key columns in adata.obs:")
print(f"  - cell_type_scanvi_filt_original: Original predictions")
print(f"  - cell_type_scanvi_filt: Corrected labels (input to scANVI)")
print(f"  - cell_type_scanvi_corrected: New scANVI predictions")
print(f"  - scanvi_confidence_corrected: Prediction confidence")

print(f"\n✅ Key matrices in adata.obsm:")
print(f"  - X_scanvi_corrected: Corrected scANVI latent space")
print(f"  - X_umap_scanvi_corrected: UMAP on corrected latent")

print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Review scanvi_correction_comparison.pdf")
print("2. Validate Memory B cell markers (CD27, IGHG1/2/3/4)")
print("3. Check if B cell subtypes are properly separated")
print("4. Proceed with downstream analysis using:")
print("   - adata.obs['cell_type_scanvi_corrected']")
print("   - adata.obsm['X_umap_scanvi_corrected']")
print("="*80)

In [ ]:
# ============================================================================
# CELL: Tissue-based visualization
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("TISSUE-BASED VISUALIZATION")
print("="*80)

# Check tissue column name
tissue_col_candidates = ['tissue', 'Tissue', 'tissue_type', 'origin', 'sample_type']
TISSUE_COL = None
for col in tissue_col_candidates:
    if col in adata.obs.columns:
        TISSUE_COL = col
        print(f"\n✓ Found tissue column: '{col}'")
        break

if TISSUE_COL is None:
    print("\n⚠️  Tissue column not found. Available columns:")
    print(adata.obs.columns.tolist())
    # Use batch as fallback
    TISSUE_COL = BATCH_KEY
    print(f"\n  Using '{TISSUE_COL}' as tissue proxy")

# Use corrected annotations
CELLTYPE_COL = 'cell_type_scanvi_corrected'
UMAP_BASIS = 'umap_scanvi_corrected'

print(f"\nTissue distribution:")
print(adata.obs[TISSUE_COL].value_counts())

# ============================================================================
# 1. UMAP colored by tissue and cell type
# ============================================================================

print(f"\n{'='*60}")
print("UMAP Visualization")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# UMAP by tissue
sc.pl.embedding(
    adata,
    basis=UMAP_BASIS,
    color=TISSUE_COL,
    ax=axes[0],
    show=False,
    title='B Cells by Tissue',
    size=3,
    palette='tab10',
    frameon=False
)

# UMAP by cell type
sc.pl.embedding(
    adata,
    basis=UMAP_BASIS,
    color=CELLTYPE_COL,
    ax=axes[1],
    show=False,
    title='B Cells by Cell Type (Corrected)',
    size=3,
    legend_loc='right margin',
    frameon=False
)

plt.tight_layout()
output_fig1 = FIG_DIR / 'umap_tissue_celltype.png'
plt.savefig(output_fig1, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {output_fig1}")

# ============================================================================
# 2. Cell type composition across tissues
# ============================================================================

print(f"\n{'='*60}")
print("Cell Type Composition Across Tissues")
print("="*60)

# Calculate proportions
prop_df = adata.obs.groupby([TISSUE_COL, CELLTYPE_COL]).size().unstack(fill_value=0)
prop_pct = prop_df.div(prop_df.sum(axis=1), axis=0) * 100

# Figure: Stacked bar + Heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Stacked bar plot
prop_pct.plot(kind='bar', stacked=True, ax=axes[0], colormap='tab20', width=0.8)
axes[0].set_title('B Cell Composition Across Tissues (%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tissue', fontsize=12)
axes[0].set_ylabel('Percentage (%)', fontsize=12)
axes[0].legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[0].set_ylim(0, 100)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')

# Heatmap
sns.heatmap(prop_pct.T, annot=True, fmt='.1f', cmap='YlOrRd', 
            cbar_kws={'label': 'Percentage (%)'}, ax=axes[1],
            linewidths=0.5, linecolor='gray')
axes[1].set_title('Cell Type Distribution Heatmap', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Tissue', fontsize=12)
axes[1].set_ylabel('Cell Type', fontsize=12)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
output_fig2 = FIG_DIR / 'tissue_celltype_composition.png'
plt.savefig(output_fig2, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved: {output_fig2}")

# Print summary table
print(f"\n{'='*60}")
print("Cell Type Proportion by Tissue (%)")
print("="*60)
print(prop_pct.round(1).to_string())

# Cell counts
print(f"\n{'='*60}")
print("Cell Counts by Tissue")
print("="*60)
print(prop_df.to_string())

print(f"\n✓ Tissue visualization complete")